# Censored Data HAL Density Estimation (Refactored EMStage)

This notebook demonstrates HAL-based density estimation for right-censored data on [0,1], using the **refactored `EMStage` architecture** where the EM algorithm is separated from the initial IPCW estimator.

## Overview

For right-censored data where we observe `(T, Δ)` with:
- `T = min(X, C)` (observed time)
- `Δ = I(X ≤ C)` (event indicator)

We implement:
1. **IPCW-HAL-MLE**: Inverse probability of censoring weighted HAL on uncensored observations
2. **EMStage Refinement**: EM with multiple imputation using IPCW as initial working model
3. **Optuna-based hyperparameter selection**: Cross-validated lambda/norm_constraint tuning

## Key Difference from Original

Instead of using `EMIPCWEstimator` which bundles IPCW + EM together, this notebook uses:
1. `WeightedCVXPYEstimator` for the initial IPCW estimate
2. `EMStage.run()` to refine the estimate with EM

This separation provides more flexibility and control over the estimation pipeline.

## Sections
1. Data Simulation (Truncated Normal)
2. Kaplan-Meier Censoring Survival Estimation
3. IPCW-HAL-MLE Initialization
4. EMStage Refinement with Multiple Imputation
5. Hyperparameter Tuning with Optuna
6. Comparison and Evaluation (KL divergence, incomplete-data log-likelihood)


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import truncnorm

# HALDensity censored-data subpackage
from haldensity.censoring import (
    KaplanMeier,
    compute_ipcw_weights,
    WeightedCVXPYEstimator,
    EMStage,              # NEW: Standalone EM stage
    EMStageResult,        # NEW: Result container
    EMIPCWEstimator,      # For comparison
    CensoredOptunaHyperparameterTuner,
    pipelines,
    kl_divergence,
    incomplete_loglik,
    mi_complete_loglik,
)

print("✓ Imports successful (including EMStage)")


## 1. Data Simulation

Generate right-censored data from a truncated normal distribution on [0,1]:
- True event times `X ~ TruncatedNormal(μ=0.5, σ=0.1, support=[0,1])`
- Censoring times `C ~ Uniform(0,1)`
- Observed: `T = min(X, C)` and `Δ = I(X ≤ C)`


In [ ]:
# Simulation parameters
np.random.seed(42)
rng = np.random.default_rng(42)

n_samples = 1000
mean, std = 0.5, 0.1
lower, upper = 0.0, 1.0

# Generate truncated normal event times
a, b = (lower - mean) / std, (upper - mean) / std
X_true = truncnorm.rvs(a, b, loc=mean, scale=std, size=n_samples, random_state=rng)

# Generate uniform censoring times
C = rng.uniform(lower, upper, size=n_samples)

# Observed data
T = np.minimum(X_true, C)
Delta = (X_true <= C).astype(int)

# Store in DataFrame
data = pd.DataFrame({"T": T, "Delta": Delta})

print(f"Sample size: {n_samples}")
print(f"Censoring rate: {(1 - Delta.mean()):.1%}")
print(f"Observed events: {Delta.sum()}")
print(f"Censored observations: {(1 - Delta).sum()}")


In [ ]:
# Visualize observed data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of observed times
axes[0].hist(data.loc[data['Delta'] == 1, 'T'], bins=30, alpha=0.6, label='Events', density=True)
axes[0].hist(data.loc[data['Delta'] == 0, 'T'], bins=30, alpha=0.6, label='Censored', density=True)
axes[0].set_xlabel('Observed Time T')
axes[0].set_ylabel('Density')
axes[0].set_title('Observed Data Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# True density
grid = np.linspace(0, 1, 500)
true_density = truncnorm.pdf(grid, a, b, loc=mean, scale=std)
axes[1].plot(grid, true_density, 'k-', linewidth=2, label='True Density')
axes[1].hist(X_true, bins=30, alpha=0.3, density=True, label='True Events (unobserved)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Density')
axes[1].set_title('True Event Time Distribution')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 2. Kaplan-Meier Censoring Survival Estimation

Estimate the censoring survival function $S_c(t) = P(C > t)$ using Kaplan-Meier on the censoring process.


In [ ]:
# Fit Kaplan-Meier for censoring survival
km = KaplanMeier()
km.fit(data, time_col="T", delta_col="Delta")

# Get step function
km_times, km_surv = km.stepwise_survival_()

print(f"KM estimated at {len(km_times)} unique time points")
print(f"S_c(0) = {km.predict(0.0):.3f}")
print(f"S_c(0.5) = {km.predict(0.5):.3f}")
print(f"S_c(1.0) = {km.predict(1.0):.3f}")


In [ ]:
# Plot KM censoring survival
plt.figure(figsize=(8, 5))
plt.step(km_times, km_surv, where='post', linewidth=2, label='KM $S_c(t)$')
plt.xlabel('Time t')
plt.ylabel('Censoring Survival Probability')
plt.title('Kaplan-Meier Estimate of Censoring Survival')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Compute IPCW weights using numpy arrays
T_vals = np.asarray(data["T"].values, dtype=float)
Delta_vals = np.asarray(data["Delta"].values, dtype=int)

# Create S_c prediction wrapper that returns numpy array
def S_c_predict(t):
    """Predict censoring survival - returns np.ndarray."""
    return np.atleast_1d(km.predict(t))

ipcw_weights = compute_ipcw_weights(
    T=T_vals,
    Delta=Delta_vals,
    S_c_predict=S_c_predict,
)

print(f"\nIPCW weights: min={ipcw_weights.min():.2f}, max={ipcw_weights.max():.2f}, mean={ipcw_weights.mean():.2f}")
print(f"Non-zero weights (events): {(ipcw_weights > 0).sum()}")


## 3. IPCW-HAL-MLE Initialization

Fit a weighted HAL density estimator using only the uncensored observations with IPCW weights. This provides a baseline and **serves as the initial working model** for the EMStage refinement.


In [ ]:
# Fit IPCW-weighted HAL estimator (order 2)
# Create DataFrame with W1 column for uncensored observations only
uncensored_mask = Delta_vals == 1
ipcw_data = pd.DataFrame({"W1": T_vals[uncensored_mask]})
ipcw_weights_unc = ipcw_weights[uncensored_mask]

# Fit the weighted estimator - this will be our INITIAL WORKING MODEL for EMStage
ipcw_estimator = WeightedCVXPYEstimator(
    norm_constraint=100,
    basis_order=2,
    solver="ECOS",
)
ipcw_estimator.fit(ipcw_data, sample_weights=ipcw_weights_unc)

# Get results
ipcw_results = ipcw_estimator.get_results()
print(f"IPCW-HAL knots selected: {len(ipcw_results['grid_points_hal_selected'])}")
print(f"IPCW-HAL coefficients: {np.array(ipcw_results['theta_hat']).shape}")

# Evaluate on grid
eval_grid = np.linspace(0, 1, 500)
ipcw_density = ipcw_estimator.get_density_at_points(eval_grid)

# Compute incomplete-data log-likelihood
ipcw_ll = incomplete_loglik(ipcw_estimator, data, time_col="T", delta_col="Delta")
print(f"IPCW incomplete-data log-likelihood: {ipcw_ll:.4f}")

# Compute KL divergence from true density
ipcw_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=ipcw_density,
)
print(f"IPCW KL divergence: {ipcw_kl:.6f}")


In [ ]:
# Plot IPCW estimate vs. true density
plt.figure(figsize=(10, 6))
plt.plot(grid, true_density, 'k-', linewidth=2, label='True Density', alpha=0.8)
plt.plot(eval_grid, ipcw_density, 'b-', linewidth=2, label=f'IPCW-HAL (KL={ipcw_kl:.4f})', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density f(x)')
plt.title('IPCW-HAL-MLE vs. True Density (Initial Working Model)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 4. EMStage Refinement with Multiple Imputation

Instead of using EMIPCWEstimator which combines IPCW + EM, we use EMStage.run() to refine the IPCW estimate via EM iterations with multiple imputation.


In [ ]:
# Create EMStage with EM configuration
em_stage = EMStage(
    m_imputations=30,          # Number of imputations per censored observation
    max_em_iter=10,            # Maximum EM iterations
    em_tol=1e-4,               # Convergence tolerance
    norm_constraint=100,       # L1 norm constraint for M-step
    n_grid_points=200,
    m_step_solver="ECOS",
    verbose=True,              # Print progress
    rng_seed=42,
)

print("EMStage configured. Running EM refinement with IPCW as initial model...")


In [ ]:
# Run EMStage with the IPCW estimator as initial working model
em_result = em_stage.run(
    initial_estimator=ipcw_estimator,  # Our IPCW estimate as input
    data=data,
    S_c_predict=S_c_predict,
)

# Get the refined estimator
em_estimator = em_result.final_estimator

print(f"\n{'='*60}")
print(f"EMStage Results:")
print(f"  EM iterations: {em_result.em_iterations}")
print(f"  EM converged: {em_result.em_converged}")
print(f"  Theta path length: {len(em_result.theta_path)}")


In [ ]:
# Evaluate the EM-refined estimator
em_density = em_estimator.get_density_at_points(eval_grid)

# Compute incomplete-data log-likelihood
em_ll = incomplete_loglik(em_estimator, data, time_col="T", delta_col="Delta")
print(f"EM incomplete-data log-likelihood: {em_ll:.4f}")

# Compute KL divergence
em_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=em_density,
)
print(f"EM KL divergence: {em_kl:.6f}")

# Compare with IPCW (initial model)
print(f"\nImprovement over IPCW (initial working model):")
print(f"  LL change: {em_ll - ipcw_ll:+.4f}")
print(f"  KL change: {em_kl - ipcw_kl:+.6f} (negative is better)")


In [ ]:
# Plot comparison: IPCW (initial) vs EMStage-refined vs True
plt.figure(figsize=(10, 6))
plt.plot(grid, true_density, 'k-', linewidth=2.5, label='True Density', alpha=0.8)
plt.plot(eval_grid, ipcw_density, 'b--', linewidth=2, label=f'IPCW-HAL Initial (KL={ipcw_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, em_density, 'r-', linewidth=2, label=f'EMStage-Refined (KL={em_kl:.4f})', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density f(x)')
plt.title('Density Estimates: IPCW (Initial) vs EMStage-Refined')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Two-Stage Hyperparameter Tuning

The new `TwoStageCensoredTuner` separates hyperparameter tuning into two stages:

**Stage 1 (Fast)**: Tune IPCW parameters (`norm_constraint`, `basis_order`) without EM
**Stage 2 (Focused)**: Tune only `m_step_norm_multiplier` using EMStage

This dramatically reduces tuning time by avoiding EM in Stage 1.


In [ ]:
# Import the two-stage tuner
from haldensity.censoring import TwoStageCensoredTuner

# Create the tuner
tuner = TwoStageCensoredTuner(
    data=data,
    cv_folds=5,
    random_state=42,
    n_grid_points=200,
    stage1_param_ranges={
        "norm_constraint": {"low": 10.0, "high": 500.0, "log": True},
        "basis_order": [0, 1, 2],
    },
    stage2_param_ranges={
        "m_step_norm_multiplier": {"low": 0.5, "high": 10.0, "log": True},
    },
    em_defaults={
        "m_imputations": 20,
        "max_em_iter": 10,
        "em_tol": 1e-3,
    },
    silent=False,
)

print("TwoStageCensoredTuner configured")


In [ ]:
# Run two-stage optimization
import time

start_time = time.time()
best_params = tuner.optimize(
    n_trials_stage1=15,  # Fast IPCW-only trials
    n_trials_stage2=10,  # Fewer EM trials needed
)
elapsed = time.time() - start_time

print(f"\nTotal optimization time: {elapsed:.1f}s")


In [ ]:
# Fit the final model with best parameters
print("Fitting final model with best parameters...")
tuned_result = tuner.fit_best_model()
tuned_estimator = tuned_result.final_estimator

print(f"\nFinal model:")
print(f"  EM iterations: {tuned_result.em_iterations}")
print(f"  EM converged: {tuned_result.em_converged}")


In [ ]:
# Evaluate the tuned model
tuned_density = tuned_estimator.get_density_at_points(eval_grid)
tuned_ll = incomplete_loglik(tuned_estimator, data, time_col="T", delta_col="Delta")
tuned_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=tuned_density,
)

print(f"Two-Stage Tuned Model:")
print(f"  Incomplete-data log-likelihood: {tuned_ll:.4f}")
print(f"  KL divergence: {tuned_kl:.6f}")

print(f"\nComparison with manual EMStage (norm_constraint=100):")
print(f"  LL difference: {tuned_ll - em_ll:+.4f}")
print(f"  KL difference: {tuned_kl - em_kl:+.6f}")


In [ ]:
# Plot all estimates
plt.figure(figsize=(12, 7))
plt.plot(grid, true_density, 'k-', linewidth=3, label='True Density', alpha=0.9)
plt.plot(eval_grid, ipcw_density, 'b--', linewidth=2, label=f'IPCW Initial (KL={ipcw_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, em_density, 'r-', linewidth=2, label=f'EMStage Manual (KL={em_kl:.4f})', alpha=0.7)
plt.plot(eval_grid, tuned_density, 'g-', linewidth=2.5, label=f'Two-Stage Tuned (KL={tuned_kl:.4f})', alpha=0.8)
plt.xlabel('x', fontsize=12)
plt.ylabel('Density f(x)', fontsize=12)
plt.title('Density Estimates: IPCW, EMStage, and Two-Stage Tuned', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Standalone EMStageTuner

The `EMStageTuner` allows you to tune EM parameters when you already have a pre-fitted initial estimator. This is useful when:
- You have a fixed IPCW estimator and only want to optimize the EM refinement
- You want to experiment with different EM configurations without re-fitting the initial model


In [ ]:
# Import EMStageTuner
from haldensity.censoring import EMStageTuner

# Use our already-fitted IPCW estimator from Section 3
# (ipcw_estimator was fitted earlier in the notebook)

em_tuner = EMStageTuner(
    data=data,
    initial_estimator=ipcw_estimator,  # Our pre-fitted IPCW model
    S_c_predict=S_c_predict,           # Censoring survival function
    cv_folds=5,
    random_state=42,
    param_ranges={
        "m_step_norm_multiplier": {"low": 0.5, "high": 10.0, "log": True},
    },
    em_defaults={
        "m_imputations": 20,
        "max_em_iter": 10,
    },
    silent=False,
)

print("EMStageTuner configured with pre-fitted IPCW estimator")


In [ ]:
# Run EMStageTuner optimization
em_tuner_start = time.time()
em_best_params = em_tuner.optimize(n_trials=10)
em_tuner_elapsed = time.time() - em_tuner_start

print(f"\nEMStageTuner optimization time: {em_tuner_elapsed:.1f}s")


In [ ]:
# Fit the best model from EMStageTuner
print("Fitting model with best EM parameters...")
em_tuner_result = em_tuner.fit_best_model()
em_tuner_estimator = em_tuner_result.final_estimator

# Evaluate
em_tuner_density = em_tuner_estimator.get_density_at_points(eval_grid)
em_tuner_ll = incomplete_loglik(em_tuner_estimator, data, time_col="T", delta_col="Delta")
em_tuner_kl = kl_divergence(
    true_pdf_fn=lambda x: truncnorm.pdf(x, a, b, loc=mean, scale=std),
    grid=eval_grid,
    est_density=em_tuner_density,
)

print(f"\nEMStageTuner Results:")
print(f"  Best m_step_norm_multiplier: {em_best_params['m_step_norm_multiplier']:.4f}")
print(f"  EM iterations: {em_tuner_result.em_iterations}")
print(f"  Incomplete-data LL: {em_tuner_ll:.4f}")
print(f"  KL divergence: {em_tuner_kl:.6f}")


In [ ]:
# Summary comparison of all methods
print("=" * 70)
print("SUMMARY: Comparison of All Methods")
print("=" * 70)

results_summary = pd.DataFrame({
    "Method": [
        "IPCW (Initial)",
        "EMStage (Manual, mult=1.0)",
        "TwoStageTuner",
        "EMStageTuner",
    ],
    "Log-Likelihood": [ipcw_ll, em_ll, tuned_ll, em_tuner_ll],
    "KL Divergence": [ipcw_kl, em_kl, tuned_kl, em_tuner_kl],
})

print(results_summary.to_string(index=False))
print("=" * 70)


In [ ]:
# Final visualization with all methods
plt.figure(figsize=(14, 8))
plt.plot(grid, true_density, 'k-', linewidth=3, label='True Density', alpha=0.9)
plt.plot(eval_grid, ipcw_density, 'b--', linewidth=2, label=f'IPCW (KL={ipcw_kl:.4f})', alpha=0.5)
plt.plot(eval_grid, em_density, 'c-', linewidth=2, label=f'EMStage Manual (KL={em_kl:.4f})', alpha=0.6)
plt.plot(eval_grid, tuned_density, 'r-', linewidth=2, label=f'TwoStageTuner (KL={tuned_kl:.4f})', alpha=0.7)
plt.plot(eval_grid, em_tuner_density, 'g-', linewidth=2.5, label=f'EMStageTuner (KL={em_tuner_kl:.4f})', alpha=0.8)
plt.xlabel('x', fontsize=12)
plt.ylabel('Density f(x)', fontsize=12)
plt.title('All Density Estimation Methods Comparison', fontsize=14)
plt.legend(fontsize=10, loc='upper right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Summary

This notebook demonstrated the refactored EMStage architecture with multiple tuning options:

### Available Tuners

| Tuner | Use Case | Speed |
|-------|----------|-------|
| `EMStageTuner` | Tune EM params with pre-fitted initial estimator | Medium |
| `TwoStageCensoredTuner` | Full pipeline: IPCW tuning + EM tuning | Fast then Medium |
| `CensoredOptunaHyperparameterTuner` | Joint tuning (original) | Slow |

### When to Use Which

- **EMStageTuner**: You have a fixed IPCW estimator, want to optimize EM refinement only
- **TwoStageCensoredTuner**: Fresh start, want efficient two-stage optimization
- **CensoredOptunaHyperparameterTuner**: Need full joint optimization (legacy)
